# NB4 - SkillOpt: Train the Skill Document Like a Neural Net

**Workshop: Self-Evolving Agents by Optimizing the Harness (no GPU)**

NB3 built a skill *library*, but it never **optimized** it: we curated with a
one-shot rubric and a hand-set threshold, then hoped. NB4 turns that into a proper
**training loop**. The mental model of the whole workshop, made literal:

> **Reflection is the gradient, the skill document is the parameter vector, and
> your eval set is the loss.**

So we run **gradient descent in text space**:

| Neural net | SkillOpt (this notebook) |
|---|---|
| parameter vector theta | the **skill document** we inject (C <- S) |
| loss | **error on a held-out validation split** |
| gradient | a **reflection** that proposes a skill *edit* from a failure |
| learning rate | how many edits we try per step |
| **validation gate** | accept an edit only if it does **not** hurt val |
| momentum | the document **persists and compounds** across steps |

The **validation gate** is the star: it is exactly what stops the 25%-degrade
trap from NB3. We log the run to **Weights & Biases**, like a real fine-tune -
except the only thing changing is text, and there is no GPU in sight.

In [ ]:
import sys, os, json, re
sys.path.insert(0, os.path.abspath(".."))
from workshop_utils import (
    build_db, load_tasks, run_sql, score_sql, evaluate,
    llm, METER, SCHEMA_TEXT, extract_sql, baseline_prompt, make_agent,
    preflight, flush,
)
preflight("WANDB_API_KEY")    # OPENAI + LANGFUSE + W&B (see SETUP.md)
build_db()

# Data discipline: TEST stays held out for the final number. We split the 24
# TRAIN tasks into an optimization set (where we look for failures to learn from)
# and a small VALIDATION set (the gate that decides which edits we keep).
train = [t for t in load_tasks() if t["split"] == "train"]
val   = train[::4]                          # every 4th train task -> the gate
opt   = [t for t in train if t not in val]
print(f"opt set: {len(opt)} tasks   val set: {len(val)} tasks   (test held out)")

try:
    baseline_acc = json.load(open("../data/baseline_test.json"))["accuracy"]
except FileNotFoundError:
    baseline_acc = evaluate(make_agent(), split="test")["accuracy"]
print("NB1 baseline test accuracy:", round(baseline_acc, 3))

## The parameter and the gradient

`theta` (our parameter) is just a **list of skills**, injected through the same
`extra=` slot NB2/NB3 used - the frozen agent never changes. We need three pieces:
a way to **measure** a skill document (forward pass + loss), and a way to turn a
failure into a **proposed edit** (the gradient).

In [ ]:
def format_skills(skills):
    if not skills:
        return ""
    lines = ["Relevant skills (reusable SQL patterns learned from past tasks):"]
    for s in skills:
        lines.append(f"- {s['name']}: USE WHEN {s['when_to_use']}. PATTERN: {s['pattern']}")
    return "\n".join(lines)

def evaluate_theta(theta, tasks):
    # One forward pass: return (accuracy, failures) for skill document `theta`.
    agent = make_agent(extra=format_skills(theta))
    fails, correct = [], 0
    for t in tasks:
        sql = agent(t["question"])
        if score_sql(sql, t["gold"]):
            correct += 1
        else:
            fails.append({"question": t["question"], "sql": sql, "gold": t["gold"]})
    return correct / len(tasks), fails

def val_accuracy(theta):
    return evaluate_theta(theta, val)[0]      # the validation "loss" (as accuracy)

PROPOSE_SYS = (
    "You improve a text-to-SQL agent by writing ONE reusable SKILL that would have "
    "prevented a specific failure. Return STRICT JSON with keys name, when_to_use, "
    "pattern. 'when_to_use' triggers a CLASS of questions (no specific values); "
    "'pattern' is the general SQL technique. Generalize - never mention the question."
)

def parse_json_obj(raw):
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

def propose_skill(failure):                    # the "gradient": failure -> edit
    raw = llm([
        {"role": "system", "content": PROPOSE_SYS},
        {"role": "user", "content":
            "Schema:\n" + SCHEMA_TEXT +
            "\nQuestion: " + failure["question"] +
            "\n\nAgent's wrong SQL:\n" + failure["sql"] +
            "\n\nCorrect SQL:\n" + failure["gold"] +
            "\n\nWrite ONE general skill (JSON) that fixes this class of mistake."},
    ])
    d = parse_json_obj(raw)
    if not d or not all(k in d for k in ("name", "when_to_use", "pattern")):
        return None
    return {k: str(d[k]) for k in ("name", "when_to_use", "pattern")}

## The optimizer: learning rate + validation gate + momentum

Each step is one round of "SGD":
1. **Forward pass** - run the current `theta` on the opt set; collect failures.
2. **Gradient** - turn up to `LR` failures into candidate skill edits.
3. **Validation gate** - apply each candidate and keep it **only if val accuracy
   does not drop**. A candidate that would pollute the prompt is rejected here,
   before it can ever hurt the held-out test number.
4. **Momentum** - accepted skills stay in `theta` and compound into the next step.

We also stash *every* proposal (kept or not) so we can run the no-gate ablation.

In [ ]:
import wandb

N_STEPS = 4          # optimization steps ("epochs")
LR      = 2          # candidate skill edits proposed per step (the learning rate)

run = wandb.init(project="rl-agents-workshop", name="nb4-skillopt",
                 config={"model": "gpt-4o-mini", "n_steps": N_STEPS, "lr": LR,
                         "n_opt": len(opt), "n_val": len(val)})
wandb.define_metric("step")
wandb.define_metric("train_acc", step_metric="step")
wandb.define_metric("val_acc", step_metric="step")

theta = []                    # the skill document (parameter) - starts empty
all_candidates = []           # every proposal, for the no-gate ablation later
curve = []
METER.reset()
for step in range(N_STEPS):
    train_acc, fails = evaluate_theta(theta, opt)        # forward pass
    vacc = val_accuracy(theta)
    curve.append((step, train_acc, vacc, len(theta)))
    wandb.log({"step": step, "train_acc": train_acc, "val_acc": vacc,
               "n_skills": len(theta)})
    print(f"step {step}: train={train_acc:.2f}  val={vacc:.2f}  skills={len(theta)}")
    if not fails:
        print("  no failures left on the opt set -> converged"); break
    for f in fails[:LR]:                                  # gradient: LR edits
        cand = propose_skill(f)
        if not cand:
            continue
        all_candidates.append(cand)
        trial_val = val_accuracy(theta + [cand])          # the validation gate
        if trial_val >= vacc:
            theta.append(cand); vacc = trial_val
            print(f"  KEEP   {cand['name']:<26} val -> {trial_val:.2f}")
        else:
            print(f"  REJECT {cand['name']:<26} val {trial_val:.2f} < {vacc:.2f} (would pollute)")

print(f"\nfinal skill document: {len(theta)} skills (from {len(all_candidates)} proposed)")
print(METER)

In [ ]:
import matplotlib.pyplot as plt
xs = [c[0] for c in curve]
plt.figure(figsize=(6, 4))
plt.plot(xs, [c[1] for c in curve], marker="o", label="train acc (opt set)")
plt.plot(xs, [c[2] for c in curve], marker="s", label="val acc (the gate)")
plt.axhline(baseline_acc, ls="--", color="gray", label="NB0 baseline (test)")
plt.xlabel("optimization step"); plt.ylabel("accuracy"); plt.ylim(0, 1)
plt.title("SkillOpt: training the skill document (weights frozen)")
plt.legend(); plt.show()

## Ablation - turn the gate off and watch it pollute

The gate is the only thing separating "learning" from "accumulating junk". Drop
it - accept **every** proposed edit, the unvetted pool from NB3 - and compare the
held-out **test** number against the gated document.

In [ ]:
theta_ungated = all_candidates                # keep everything we ever proposed

METER.reset()
test_gated   = evaluate(make_agent(extra=format_skills(theta)),         split="test")["accuracy"]
test_ungated = evaluate(make_agent(extra=format_skills(theta_ungated)), split="test")["accuracy"]
print("baseline (no skills)        :", round(baseline_acc, 3))
print("SkillOpt + gate  (NB4)      :", round(test_gated, 3),   f"  [{len(theta)} skills]")
print("accept-everything (no gate) :", round(test_ungated, 3), f"  [{len(theta_ungated)} skills]")
print(METER)

wandb.summary["baseline_acc"]   = baseline_acc
wandb.summary["test_gated"]     = test_gated
wandb.summary["test_ungated"]   = test_ungated
wandb.summary["n_skills_final"] = len(theta)
wandb.finish()

In [ ]:
import matplotlib.pyplot as plt
labels = ["baseline\n(NB0)", "SkillOpt\n+ gate", "no gate\n(accept all)"]
vals = [baseline_acc, test_gated, test_ungated]
plt.figure(figsize=(6, 4))
bars = plt.bar(labels, vals, color=["gray", "seagreen", "indianred"])
plt.ylim(0, 1); plt.ylabel("test accuracy")
plt.title("The validation gate is what makes optimization safe")
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center")
plt.show()
flush()

## Takeaways

- **Optimizing a prompt is training.** We ran SGD with a parameter (the skill
  document), a loss (validation error), a gradient (reflection), a learning rate,
  and momentum - no weights, no GPU.
- The **validation gate** is the load-bearing idea: it turns "self-improvement"
  from a hope into a guarantee that each accepted edit was *measured* not to hurt.
- Tracked in **W&B** like any training run, so the curve, the baseline, and the
  gated-vs-ungated result are all on one shareable run.

### The gap this leaves (-> NB5)
Our document is **flat** - one list, every skill equal, retrieved the same way.
Real libraries are **hierarchical** (broad strategies -> specific patterns), and
the best skills are often written by a **strong** model and *transferred* to a
weaker, cheaper one. That is NB5.

### Exercise
1. Shrink `val` to 2 tasks. Does the gate get noisier (more bad keeps)? You just
   felt validation-set variance - the same bias/variance trade-off as in real ML.
2. Raise `LR` to 4. More edits per step costs more API calls (watch the meter) -
   does test accuracy actually improve, or do you just overfit the opt set?
3. Make the gate strict (`trial_val > vacc`). Fewer skills survive - compare the
   final test number and the skill count.